###  np.cumprod

In [1]:
import itertools
        

In [2]:
def get_shape(a):
    if not isinstance(a , list): 
        return ()
    if len(a) == 0: 
        return (0 , )
    inner_shape = get_shape(a[0])

    for i in range(1 , len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array detected : the shape of first row is {inner_shape}" , 
                f"but row {i} has shape {get_shape(a[i])}"
            )

    return (len(a) , ) + inner_shape

def get_ndim(a):
    return len(get_shape(a))

def flatten(a): 
    result = []
    if not isinstance(a , list):
        return [a]
    for item in a:
        result.extend(flatten(item))
    return result

def deep_copy(a):
    if isinstance(a , list):
        return [deep_copy(item) for item in a]
    return a 

def cast_all(a , dtype):
    if isinstance( a , list):
        return [cast_all(x , dtype) for x in a]

    try:
        return dtype(a)
    except ValueError:
        try:
            return dtype(float(a))
        except (ValueError , TypeError): 
            raise TypeError(
                f"cannot cast element {a} to {dtype.__name__}"
            )
    except TypeError: 
        raise TypeError(
            f"cannot cast element {a} (type : {type(a).__name__})"
            f"to {dtype.__name__}"
        )

def get_element(a , indices): 
    depth = 0

    for idx in indices:
        if not isinstance(a , list): 
            raise TypeError(f"expected a list at depth {depth} , but git {type(a).__name__}")
        if idx >= len(a) or idx < -len(a):
            raise IndexError(f"at depth {depth} index {idx} is out of range for {len(a)}")

        a = a[idx]
        depth += 1

    return a

def set_element(a, indices, value):
    depth = 0
    for idx in indices[:-1]:
        if not isinstance(a, list):
            raise TypeError(f"at depth {depth}: expected list, got {type(a).__name__}")

        if idx >= len(a) or idx < -len(a):
            raise IndexError(f"at depth {depth}: index {idx} out of range")

        a = a[idx]
        depth += 1

    last_idx = indices[-1]

    if not isinstance(a, list):
        raise TypeError("too many indices")

    if last_idx >= len(a) or last_idx < -len(a):
        raise IndexError(f"final index {last_idx} out of range")

    a[last_idx] = value

In [6]:
def np_cumprod(a, axis=None, dtype=None, out=None):
    if not isinstance(a, list):
        raise ValueError(f"expected list but got {type(a).__name__}")

    shape = get_shape(a)
    ndim = len(shape)

    if ndim == 0:
        raise ValueError("np_cumprod does not support 0-d input")
    if dtype is not None:
        a = cast_all(a, dtype)
    else:
        flat = flatten(a)
        if flat and isinstance(flat[0], float):
            dtype = float
        else:
            dtype = int
        a = cast_all(a, dtype)

   
    if axis is None:
        flat = flatten(a)
        if len(flat) == 0:
            return []
        result = flat[:]
        for i in range(1, len(result)):
            result[i] = result[i] * result[i - 1]   # only change from cumsum
        if out is not None:
            if len(out) != len(result):
                raise ValueError("out shape doesn't match result shape")
            for i in range(len(result)):
                out[i] = result[i]
            return out
        return result

    if axis < -ndim or axis >= ndim:
        raise IndexError(
            f"axis {axis} is out of bounds for array of dimension {ndim}"
        )
    if axis < 0:
        axis = ndim + axis

 
    if 0 in shape:
        return deep_copy(a)

   
    result = deep_copy(a)

    other_dims = []
    for i in range(ndim):
        if i != axis:
            other_dims.append(range(shape[i]))

    for combo in itertools.product(*other_dims):
        for ax_i in range(1, shape[axis]):

            current_idx = list(combo)
            current_idx.insert(axis, ax_i)

            prev_idx = list(combo)
            prev_idx.insert(axis, ax_i - 1)

            current_value = get_element(result, current_idx)
            previous_value = get_element(result, prev_idx)

            new_value = current_value * previous_value   # only change from cumsum
            set_element(result, current_idx, new_value)

 
    if out is not None:
        if get_shape(out) != get_shape(result):
            raise ValueError("out shape doesn't match result shape")
        for combo in itertools.product(*[range(s) for s in shape]):
            set_element(out, list(combo), get_element(result, list(combo)))
        return out

    return result

### test_cases :

In [9]:
print(np_cumprod([1, 2, 3, 4]))   
print(np_cumprod([3, 0, 5, 2]))    
print(np_cumprod([1.5, 2.0, 4.0]) )
print(np_cumprod([-1, -2, -3])) 
print(np_cumprod([1, 2, 3], dtype=float)) 
print(np_cumprod([[1, 2], [3, 4]]))  
print(np_cumprod([[1, 2], [3, 4]], axis=None))  
print(np_cumprod([[1, 2], [3, 4]], axis=0)) 
print(np_cumprod([[1, 2, 3], [4, 5, 6]], axis=1))     
print(np_cumprod([[2], [3], [4]], axis=0)) 
print(np_cumprod([[1, 2, 3], [4, 5, 6]], axis=-1))
print(np_cumprod([[1, 2], [3, 4]], axis=-2)) 

[1, 2, 6, 24]
[3, 0, 0, 0]
[1.5, 3.0, 12.0]
[-1, 2, -6]
[1.0, 2.0, 6.0]
[1, 2, 6, 24]
[1, 2, 6, 24]
[[1, 2], [3, 8]]
[[1, 2, 6], [4, 20, 120]]
[[2], [6], [24]]
[[1, 2, 6], [4, 20, 120]]
[[1, 2], [3, 8]]


In [10]:
np_cumprod([1, 2, 3], axis=2)

IndexError: axis 2 is out of bounds for array of dimension 1

In [11]:
np_cumprod([[1, 2], [3, 4]], axis=5) 

IndexError: axis 5 is out of bounds for array of dimension 2

In [12]:
np_cumprod([1, "two", 3])   

TypeError: cannot cast element two to int

In [13]:
np_cumprod([[1, 2], [3]]) 

ValueError: ('Jagged array detected : the shape of first row is (2,)', 'but row 1 has shape (1,)')

In [14]:
np_cumprod(42) 

ValueError: expected list but got int